# Leader Election & Consensus

## 🧠 Mental Model

> **Leader election = an election where only one candidate can win at any time.
> If the elected leader dies mid-term, a new election is called automatically.
> Consensus = the process by which distributed nodes agree on the winner —
> and agree on the order of all decisions thereafter.**

### WHY Leader Election Exists

Many distributed operations must be performed by exactly ONE node to avoid conflicts:
- **Scheduled jobs** — run a cron job exactly once across 10 servers
- **Singleton services** — exactly one instance processes a Kafka partition
- **Distributed locks** — one process holds the lock at a time
- **Shard master** — one node coordinates writes for a shard
- **Primary DB selection** — one node accepts writes

Without leader election, either:
1. **No one acts** — work doesn't get done
2. **Everyone acts** — double processing, conflicts, data corruption


---
## Consensus Algorithms — Brief Survey

### Paxos (1989) — Theoretically important, practically painful

```
Phase 1 (Prepare):  Proposer → all acceptors: "I propose value V; accept my proposal #N?"
Phase 1 (Promise):  Acceptor → proposer: "I promise not to accept lower #N"
Phase 2 (Accept):   Proposer → all acceptors: "Please accept V with proposal #N"
Phase 2 (Accepted): Acceptor → all: "I accepted V #N"
Learners receive the accepted value.

Problems: hard to understand, hard to implement correctly, needs multi-round messages
```

### Raft (2014) — Designed for understandability

```
Raft decomposes consensus into:
1. Leader election    (who is the current leader?)
2. Log replication    (leader distributes commands to followers)
3. Safety             (committed entries never lost)

States: Follower → Candidate → Leader
  - Follower: receives heartbeats from leader
  - Candidate: election timeout → becomes candidate, requests votes
  - Leader: sends heartbeats; replicates log entries to followers
```

### 🌍 Where Used in Production

| System | Algorithm | Usage |
|---|---|---|
| etcd (Kubernetes) | Raft | Cluster state, config, secrets storage |
| Apache ZooKeeper | ZAB (Zookeeper Atomic Broadcast, Paxos-like) | Kafka leader election, Hadoop NameNode |
| CockroachDB | Raft | Per-range leader election |
| MongoDB | Raft-like | Replica set primary election |
| Consul | Raft | Service health and config consensus |


In [ ]:
import time, threading, random
from enum import Enum
from dataclasses import dataclass, field

class RaftState(Enum):
    FOLLOWER  = "follower"
    CANDIDATE = "candidate"
    LEADER    = "leader"

@dataclass
class RaftNode:
    node_id:         int
    cluster_size:    int
    state:           RaftState = RaftState.FOLLOWER
    current_term:    int = 0
    voted_for:       int | None = None
    votes_received:  int = 0
    leader_id:       int | None = None
    alive:           bool = True
    log:             list = field(default_factory=list)

    def request_vote(self, candidate_id: int, candidate_term: int) -> bool:
        '''Respond to a vote request from a candidate.'''
        if not self.alive: return False
        if candidate_term < self.current_term: return False
        if candidate_term > self.current_term:
            self.current_term = candidate_term
            self.state        = RaftState.FOLLOWER
            self.voted_for    = None
        if self.voted_for is None or self.voted_for == candidate_id:
            self.voted_for = candidate_id
            return True
        return False

    def receive_heartbeat(self, leader_id: int, term: int):
        if not self.alive: return
        if term >= self.current_term:
            self.current_term = term
            self.state        = RaftState.FOLLOWER
            self.leader_id    = leader_id
            self.voted_for    = None

class RaftCluster:
    def __init__(self, n: int):
        self.nodes = [RaftNode(i, n) for i in range(n)]
        self.quorum = n // 2 + 1

    def elect_leader(self) -> int | None:
        '''Simulate one round of Raft leader election.'''
        # Pick a candidate (node with no current leader)
        candidates = [n for n in self.nodes if n.alive and n.leader_id is None]
        if not candidates: return None
        candidate = random.choice(candidates)

        candidate.state        = RaftState.CANDIDATE
        candidate.current_term += 1
        candidate.voted_for    = candidate.node_id
        candidate.votes_received = 1  # votes for itself
        term = candidate.current_term

        print(f"  Node {candidate.node_id} starts election for term {term}")

        # Request votes from all other nodes
        for node in self.nodes:
            if node.node_id == candidate.node_id: continue
            if node.request_vote(candidate.node_id, term):
                candidate.votes_received += 1
                print(f"    Node {node.node_id} → votes YES for node {candidate.node_id}")
            else:
                print(f"    Node {node.node_id} → votes NO (already voted or higher term)")

        if candidate.votes_received >= self.quorum:
            candidate.state    = RaftState.LEADER
            candidate.leader_id = candidate.node_id
            print(f"  ✅ Node {candidate.node_id} elected LEADER "
                  f"(got {candidate.votes_received}/{len(self.nodes)} votes, quorum={self.quorum})")
            # Send heartbeat to all followers
            for node in self.nodes:
                if node.node_id != candidate.node_id:
                    node.receive_heartbeat(candidate.node_id, term)
            return candidate.node_id
        else:
            candidate.state = RaftState.FOLLOWER
            print(f"  ❌ Node {candidate.node_id} lost election "
                  f"(got {candidate.votes_received}/{self.quorum} needed)")
            return None

    def fail_node(self, node_id: int):
        self.nodes[node_id].alive = False
        if self.nodes[node_id].state == RaftState.LEADER:
            # Leader died — reset all followers
            for n in self.nodes:
                n.leader_id = None
                if n.alive and n.state == RaftState.LEADER:
                    n.state = RaftState.FOLLOWER
            print(f"  [Cluster] Node {node_id} (LEADER) FAILED → new election needed")
        else:
            print(f"  [Cluster] Node {node_id} (follower) FAILED")

    def status(self):
        print("  Cluster status:", ", ".join(
            f"N{n.node_id}={'L' if n.state==RaftState.LEADER else 'F' if n.alive else 'X'}"
            for n in self.nodes
        ))

# ── Demo ─────────────────────────────────────────────────────────────────────
print("=== Raft Leader Election Demo ===
")
cluster = RaftCluster(5)
print("Initial state:")
cluster.status()

print("
Starting election:")
leader = cluster.elect_leader()
cluster.status()

print(f"
Simulate leader (Node {leader}) failure:")
cluster.fail_node(leader)
cluster.status()

print("
Starting new election:")
new_leader = cluster.elect_leader()
cluster.status()
print(f"
Leadership transferred from Node {leader} → Node {new_leader}")


---
## Practical Leader Election Patterns

### Using Redis SETNX (Distributed Lock as Leader Election)

```python
# Simplified — in production use Redlock algorithm
def try_become_leader(redis, node_id, ttl=30):
    # SET leader_key "node_id" NX PX 30000
    result = redis.set("leader", node_id, nx=True, ex=ttl)
    return result is not None

def maintain_leadership(redis, node_id, ttl=30):
    # Extend TTL while we're the leader (before it expires)
    current = redis.get("leader")
    if current == node_id:
        redis.expire("leader", ttl)
        return True
    return False
```

### Using etcd for Production Leader Election

```python
import etcd3

client = etcd3.client()

# Create election
election = client.election("/shopflow/cron-leader")

# Blocks until this instance becomes leader
with election.campaign(value="node-1"):
    print("I am the leader!")
    run_scheduled_jobs()
# Leadership released when context exits
# Next waiting candidate automatically wins
```

### The "Thundering Herd" Problem on Leader Failure

```
Leader dies at t=0. All N-1 followers detect the failure after their
election timeout (150ms-300ms). All N-1 start an election simultaneously.
→ Votes split across multiple candidates → no winner → repeat.

Raft's fix: RANDOMIZED election timeouts (150-300ms random per node).
One node wakes up first with high probability → wins quickly.
```

### ⚠️ Gotchas

- **"Zombie leader"**: Old leader is alive but partitioned — thinks it's still leader.
  New leader elected on the other side. Now two leaders!
  Fix: **Fencing tokens** — leader gets token N; DB rejects writes from token < current.
- **etcd as SPOF**: Run 3 or 5 etcd nodes. Kubernetes uses 3 etcd nodes minimum.
- **Redis single-node leader election**: NOT safe. Redis can fail between SETNX and expire.
  Use Redlock (6 Redis nodes, majority) for production.
